In [1]:
import pandas as pd
import numpy as np
import pathlib
import os
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import pathlib

from final_project_package.ml_logic.data_clean import initialize_clip, data_clean, add_clip_columns, average_scoring, parse_layout
from final_project_package.ml_logic.model import initialize_model, train_model, evaluate_model
from final_project_package.ml_logic.preprocessor_pipeline import get_fitted_preprocessor


✅ TensorFlow loaded (0.03s)


In [2]:
#load dataset(double check name of the main dataframe)
data_path = pathlib.Path(".")
data = pd.read_csv(data_path / "data_dump/listings_with_scores.csv")
# image_data = pd.read_csv(data_path / 'raw_data/images.csv')
data


,index,source_id,url,price_man_yen,area_sqm,year_built,floor_number,floors_total,address,nearest_station,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,rooms_num,base_layout
0,0,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2390,52.89,1989,4,4,東京都足立区六月１ [ ■ 周辺環境 ],竹ノ塚駅,...,0.226562,NaN,0.132812,0.401367,0.203125,0.531250,NaN,0.246094,2,LDK
1,1,20332918,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,1580,29.16,1976,6,9,東京都足立区東和５ [ ■ 周辺環境 ],北綾瀬駅,...,0.495117,NaN,0.183594,0.378906,0.539062,0.621094,NaN,0.320312,1,DK
2,2,20344616,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2380,53.04,1974,3,8,東京都足立区梅田７-22-9 [ ■ 周辺環境 ],梅島駅,...,0.347656,NaN,0.042969,0.255208,0.462891,0.468750,NaN,0.203125,2,LDK
3,3,78385243,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_7...,1700,40.10,1985,1,5,東京都足立区青井１-16-9 [ ■ 周辺環境 ],青井駅,...,NaN,NaN,NaN,0.394531,0.634766,NaN,NaN,NaN,2,DK
4,4,78151350,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_7...,1800,26.29,1977,5,5,東京都文京区小石川５ [ ■ 周辺環境 ],茗荷谷駅,...,0.367188,NaN,NaN,0.281250,0.621094,0.658203,NaN,NaN,1,DK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3348,3348,78637031,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,118000,106.76,2024,5,2,東京都渋谷区神宮前６ [ ■ 周辺環境 ],明治神宮前駅,...,0.281250,0.287109,NaN,0.417969,0.677083,0.626953,0.733398,NaN,2,LDK
3349,3349,78783232,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,79800,152.01,2014,2,1,東京都渋谷区広尾３-12-17 [ ■ 周辺環境 ],広尾駅,...,0.377604,0.366406,NaN,NaN,NaN,0.764323,0.632031,NaN,3,LDK
3350,3350,78931650,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,137000,214.98,2024,4,1,東京都渋谷区鉢山町 [ ■ 周辺環境 ],代官山駅,...,0.562500,0.392578,0.468750,0.369141,0.816406,0.851562,0.824219,0.378906,3,LDK
3351,3351,79115894,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,79800,177.80,2025,2,4,東京都渋谷区西原３ [ ■ 周辺環境 ],代々木上原駅,...,0.531250,0.378906,NaN,0.586719,0.761719,0.638672,0.621094,NaN,2,LDK


In [3]:
X = data.drop(columns=["price_man_yen"]).copy()
y = data["price_man_yen"]

In [4]:
X

,index,source_id,url,area_sqm,year_built,floor_number,floors_total,address,nearest_station,walk_minutes,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,rooms_num,base_layout
0,0,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,52.89,1989,4,4,東京都足立区六月１ [ ■ 周辺環境 ],竹ノ塚駅,16,...,0.226562,NaN,0.132812,0.401367,0.203125,0.531250,NaN,0.246094,2,LDK
1,1,20332918,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,29.16,1976,6,9,東京都足立区東和５ [ ■ 周辺環境 ],北綾瀬駅,12,...,0.495117,NaN,0.183594,0.378906,0.539062,0.621094,NaN,0.320312,1,DK
2,2,20344616,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,53.04,1974,3,8,東京都足立区梅田７-22-9 [ ■ 周辺環境 ],梅島駅,7,...,0.347656,NaN,0.042969,0.255208,0.462891,0.468750,NaN,0.203125,2,LDK
3,3,78385243,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_7...,40.10,1985,1,5,東京都足立区青井１-16-9 [ ■ 周辺環境 ],青井駅,10,...,NaN,NaN,NaN,0.394531,0.634766,NaN,NaN,NaN,2,DK
4,4,78151350,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_7...,26.29,1977,5,5,東京都文京区小石川５ [ ■ 周辺環境 ],茗荷谷駅,9,...,0.367188,NaN,NaN,0.281250,0.621094,0.658203,NaN,NaN,1,DK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3348,3348,78637031,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,106.76,2024,5,2,東京都渋谷区神宮前６ [ ■ 周辺環境 ],明治神宮前駅,3,...,0.281250,0.287109,NaN,0.417969,0.677083,0.626953,0.733398,NaN,2,LDK
3349,3349,78783232,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,152.01,2014,2,1,東京都渋谷区広尾３-12-17 [ ■ 周辺環境 ],広尾駅,12,...,0.377604,0.366406,NaN,NaN,NaN,0.764323,0.632031,NaN,3,LDK
3350,3350,78931650,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,214.98,2024,4,1,東京都渋谷区鉢山町 [ ■ 周辺環境 ],代官山駅,10,...,0.562500,0.392578,0.468750,0.369141,0.816406,0.851562,0.824219,0.378906,3,LDK
3351,3351,79115894,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,177.80,2025,2,4,東京都渋谷区西原３ [ ■ 周辺環境 ],代々木上原駅,5,...,0.531250,0.378906,NaN,0.586719,0.761719,0.638672,0.621094,NaN,2,LDK


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
preprocesser = get_fitted_preprocessor(X_train)


Preprocessing features...
✅ returned preprocessor


In [6]:
preprocesser.get_feature_names_out()

array(['keep_columns__source_id', 'keep_columns__rooms_num',
       'keep_columns__luxury_bathroom', 'keep_columns__luxury_bedroom',
       'keep_columns__luxury_kitchen', 'keep_columns__luxury_living_room',
       'keep_columns__luxury_toilet', 'keep_columns__brightness_bathroom',
       'keep_columns__brightness_bedroom',
       'keep_columns__brightness_kitchen',
       'keep_columns__brightness_living_room',
       'keep_columns__brightness_toilet',
       'keep_columns__condition_bathroom',
       'keep_columns__condition_bedroom',
       'keep_columns__condition_kitchen',
       'keep_columns__condition_living_room',
       'keep_columns__condition_toilet', 'num_transformer__area_sqm',
       'num_transformer__year_built', 'num_transformer__floor_number',
       'num_transformer__floors_total', 'num_transformer__walk_minutes',
       'nearest_station_tranformer__nearest_station_三ノ輪駅',
       'nearest_station_tranformer__nearest_station_三軒茶屋駅',
       'nearest_station_tranformer__

In [7]:
X_train_preprocessed = pd.DataFrame(preprocesser.transform(X_train), columns=preprocesser.get_feature_names_out(), index=X_train.index)

In [8]:
X_train_preprocessed

,keep_columns__source_id,keep_columns__rooms_num,keep_columns__luxury_bathroom,keep_columns__luxury_bedroom,keep_columns__luxury_kitchen,keep_columns__luxury_living_room,keep_columns__luxury_toilet,keep_columns__brightness_bathroom,keep_columns__brightness_bedroom,keep_columns__brightness_kitchen,...,nearest_station_tranformer__nearest_station_金町駅,nearest_station_tranformer__nearest_station_長原駅,nearest_station_tranformer__nearest_station_飯田橋駅,nearest_station_tranformer__nearest_station_駒沢大学駅,nearest_station_tranformer__nearest_station_高井戸駅,nearest_station_tranformer__nearest_station_高田馬場駅,nearest_station_tranformer__nearest_station_高輪台駅,nearest_station_tranformer__nearest_station_麻布十番駅,nearest_station_tranformer__nearest_station_infrequent_sklearn,ordinal__base_layout
2963,20044222.0,2.0,0.501302,0.263672,0.415625,0.347656,NaN,0.177083,0.335938,0.235937,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
1496,20128064.0,3.0,NaN,NaN,0.679688,0.835938,NaN,NaN,NaN,0.636719,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
1158,78899976.0,2.0,0.707031,0.572917,0.632812,0.470052,0.531250,0.378906,0.428385,0.451823,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
779,78931553.0,3.0,0.500000,0.302344,0.312500,NaN,0.679688,0.187500,0.309375,0.234375,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
2441,20255526.0,1.0,0.530599,0.562500,0.468750,0.437500,0.492188,0.168620,0.468750,0.269531,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,78958091.0,1.0,0.531250,0.349609,0.453125,0.392578,0.562500,0.226562,0.392578,0.158203,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
1070,20314805.0,2.0,0.642578,0.357812,0.347656,0.544271,0.468750,0.300781,0.203906,0.134766,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
193,20253514.0,2.0,0.468750,0.341797,0.468750,NaN,NaN,0.294922,0.530273,0.292969,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
1756,78455128.0,1.0,0.517578,0.330078,0.406250,0.378906,NaN,0.094727,0.148438,0.121094,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0


In [9]:
y_train

2963    22600
1496     8180
1158     6699
779      5480
2441    14800
        ...  
59       2890
1070     6498
193      3498
1756     9480
3314    68000
Name: price_man_yen, Length: 2347, dtype: int64

In [10]:
X_train

,index,source_id,url,area_sqm,year_built,floor_number,floors_total,address,nearest_station,walk_minutes,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,rooms_num,base_layout
2963,2963,20044222,https://suumo.jp/ms/chuko/tokyo/sc_shinjuku/nc...,70.75,2015,30,55,東京都新宿区富久町 [ ■ 周辺環境 ],新宿御苑前駅,7,...,0.235937,0.320312,NaN,0.479167,0.736328,0.606250,0.796875,NaN,2,LDK
1496,1496,20128064,https://suumo.jp/ms/chuko/tokyo/sc_itabashi/nc...,68.57,2025,5,10,東京都板橋区仲町 [ ■ 周辺環境 ],大山駅,6,...,0.636719,0.320312,NaN,NaN,NaN,0.816406,0.777344,NaN,3,LDK
1158,1158,78899976,https://suumo.jp/ms/chuko/tokyo/sc_ota/nc_7889...,52.36,1994,9,14,東京都大田区大森西２ [ ■ 周辺環境 ],平和島駅,5,...,0.451823,0.462240,0.164062,0.468750,0.755208,0.378906,0.735677,0.269531,2,LDK
779,779,78931553,https://suumo.jp/ms/chuko/tokyo/sc_sumida/nc_7...,61.78,1995,4,7,東京都墨田区東向島５-34-3 [ ■ 周辺環境 ],東向島駅,8,...,0.234375,NaN,0.066406,0.439453,0.510938,0.500000,NaN,0.164062,3,LDK
2441,2441,20255526,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,48.24,2013,2,9,東京都渋谷区桜丘町 [ ■ 周辺環境 ],渋谷駅,6,...,0.269531,0.251953,0.480469,0.298177,0.867188,0.679688,0.511719,0.472656,1,LDK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,59,78958091,https://suumo.jp/ms/chuko/tokyo/sc_nerima/nc_7...,34.65,1991,4,4,東京都練馬区貫井２-23-3 [ ■ 周辺環境 ],中村橋駅,8,...,0.158203,0.234375,0.246094,0.339844,0.636719,0.453125,0.753906,0.378906,1,LDK
1070,1070,20314805,https://suumo.jp/ms/chuko/tokyo/sc_itabashi/nc...,53.41,2001,4,15,東京都板橋区中丸町 [ ■ 周辺環境 ],要町駅,11,...,0.134766,0.183594,0.121094,0.546875,0.594531,0.333984,0.591146,0.222656,2,LDK
193,193,20253514,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,62.22,1997,2,7,東京都足立区加平３ [ ■ 周辺環境 ],北綾瀬駅,8,...,0.292969,NaN,NaN,0.412109,0.553711,0.562500,NaN,NaN,2,LDK
1756,1756,78455128,https://suumo.jp/ms/chuko/tokyo/sc_shinjuku/nc...,55.48,1995,2,3,東京都新宿区中里町27 [ ■ 周辺環境 ],神楽坂駅,5,...,0.121094,0.222656,NaN,0.273438,0.658203,0.484375,0.652344,NaN,1,LDK
